# ChatEV — Interfaz Conversacional (standalone)

Carga el modelo T5 ya entrenado desde Drive y lanza la interfaz Gradio.
**No necesita reentrenar** — solo requiere que hayas ejecutado ChatEV_nuevo.ipynb
al menos una vez y guardado el modelo con la celda de guardado.

**Archivos necesarios en Drive (carpeta `chatev_model/`):**
- `chatev_model/weights/` — pesos del modelo T5 + tokenizer
- `chatev_model/dataset_arrays.npz` — arrays numpy de ocupación, precios, etc.
- `chatev_model/dataset_meta.json` — metadatos (zonas, splits, hiperparámetros)

**También necesarios en Drive (misma carpeta que los CSVs):**
- `information.csv`, `occupancy.csv`, `price.csv`, `duration.csv` (ST-EVCDP)
- `calendar_features.csv` (opcional)
- `weather_shenzhen.csv` (opcional)
- `addresses_shenzhen.csv` (opcional)

In [15]:
!pip install gradio transformers sentencepiece gdown -q

In [16]:
import os, glob, json, re
import numpy as np
import pandas as pd
import torch
import gradio as gr
from transformers import T5ForConditionalGeneration, T5Tokenizer

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

Device: cuda


In [17]:
# Auto-detectar entorno: Colab (Drive compartido) o local (Mac/Linux)
IN_COLAB = False
try:
    from google.colab import drive as _drive
    IN_COLAB = True
except ImportError:
    pass

DRIVE_FOLDER_ID = "1eedMIiny2QJVYhbLFUPWUqEnjz8kuj2q"  # carpeta pública compartida

if IN_COLAB:
    LOCAL_DOWNLOAD = "/content/chatev_data"

    # 1. Intentar Drive montado (autora del proyecto)
    owner_path = "/content/drive/MyDrive/datasets"
    if os.path.isdir("/content/drive/MyDrive") and os.path.exists(f"{owner_path}/information.csv"):
        DATASETS_PATH = owner_path
        print("Usando Drive montado (modo autora)")
    else:
        # 2. Descargar desde carpeta Drive pública
        if not os.path.exists(LOCAL_DOWNLOAD):
            print("Descargando datos desde Drive compartido (primera vez ~250 MB)...")
            import gdown
            gdown.download_folder(id=DRIVE_FOLDER_ID, output=LOCAL_DOWNLOAD, quiet=False, use_cookies=False)
        else:
            print("Datos ya descargados, saltando descarga.")

        # Localizar information.csv dentro de lo descargado
        candidates = glob.glob(f"{LOCAL_DOWNLOAD}/**/information.csv", recursive=True)
        if not candidates:
            raise FileNotFoundError(
                "information.csv no encontrado tras descarga."
                "Verifica que la carpeta Drive compartida contiene los datasets."
            )
        DATASETS_PATH = os.path.dirname(candidates[0])
        print(f"Datos encontrados en: {DATASETS_PATH}")
else:
    DATASETS_PATH = "/Users/jimenamillamoreno/Desktop/Trabajo-final-modelado/datasets"
    if not os.path.exists(f"{DATASETS_PATH}/information.csv"):
        raise FileNotFoundError(
            f"information.csv no encontrado en {DATASETS_PATH}"
            "Ajusta DATASETS_PATH en esta celda."
        )

MODEL_SAVE_DIR = os.path.join(DATASETS_PATH, "chatev_model")

print(f"Entorno:  {'Colab' if IN_COLAB else 'Local'}")
print(f"Datasets: {DATASETS_PATH}")
print(f"Modelo:   {MODEL_SAVE_DIR}")

if not os.path.isdir(f"{MODEL_SAVE_DIR}/weights"):
    raise FileNotFoundError(
        f"Modelo no encontrado en {MODEL_SAVE_DIR}/weights"
        "Verifica que la carpeta Drive compartida contiene chatev_model/weights/"
    )

Usando Drive montado (modo autora)
Entorno:  Colab
Datasets: /content/drive/MyDrive/datasets
Modelo:   /content/drive/MyDrive/datasets/chatev_model


In [18]:
# Cargar modelo + tokenizer desde Drive
print("Cargando modelo...")
model     = T5ForConditionalGeneration.from_pretrained(f"{MODEL_SAVE_DIR}/weights").to(DEVICE)
tokenizer = T5Tokenizer.from_pretrained(f"{MODEL_SAVE_DIR}/weights")
model.eval()
print(f"Modelo cargado ({sum(p.numel() for p in model.parameters()):,} parámetros)")

# Cargar metadatos
with open(f"{MODEL_SAVE_DIR}/dataset_meta.json") as f:
    meta = json.load(f)

LOOKBACK_W = meta["LOOKBACK_W"]
HORIZON    = meta["HORIZON"]
STRIDE     = meta["STRIDE"]
time_div   = {k: tuple(v) for k, v in meta["time_div"].items()}
seen_zonas = meta["seen_zonas"]
n_zonas    = meta["n_zonas"]

print(f"LOOKBACK_W={LOOKBACK_W}, HORIZON={HORIZON}, n_zonas={n_zonas}")
print(f"seen_zonas: {len(seen_zonas)} | test split: {time_div['test']}")

Cargando modelo...


Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

Modelo cargado (60,506,624 parámetros)
LOOKBACK_W=12, HORIZON=6, n_zonas=247
seen_zonas: 198 | test split: (6912, 8640)


In [19]:
# Cargar arrays numpy
arrs = np.load(f"{MODEL_SAVE_DIR}/dataset_arrays.npz")
occ_norm   = arrs["occ_norm"]    # (n_zonas, n_pasos)
vecino_occ = arrs["vecino_occ"]
price_norm = arrs["price_norm"]
duration   = arrs["duration"]

# Normalizar duration igual que en ChatEV_nuevo
dur_max = duration[:, :time_div["train"][1]].max()
dur_norm = np.clip(duration / (dur_max + 1e-8), 0, 1)

print(f"Arrays cargados: occ_norm {occ_norm.shape}")

# Cargar information.csv
info = pd.read_csv(f"{DATASETS_PATH}/information.csv", index_col=0)
info.columns = [c.strip().lower().replace(" ", "_") for c in info.columns]

# Timestamps (reconstruir desde parámetros)
timestamps = pd.date_range(start="2022-06-19 00:00:00", periods=meta["n_pasos"], freq="5min")

print(f"info: {len(info)} zonas | timestamps: {len(timestamps)} pasos")

Arrays cargados: occ_norm (247, 8640)
info: 247 zonas | timestamps: 8640 pasos


In [20]:
# Cargar features externas (opcional — el chat funciona sin ellas)
WEATHER_AVAILABLE  = False
ADDRESS_AVAILABLE  = False
SPATIAL_AVAILABLE  = False
CALENDAR_AVAILABLE = False

ext = {}  # almacén de features externas

# Weather
try:
    df_w = pd.read_csv(f"{DATASETS_PATH}/weather_shenzhen.csv")
    n_t  = meta["n_pasos"]
    t_tr = time_div["train"][1]
    def _norm_w(col):
        mn, mx = col[:t_tr].min(), col[:t_tr].max()
        return np.clip((col - mn) / (mx - mn + 1e-8), 0, 1)
    ext["temperature"]      = df_w["temperature"].values[:n_t].astype(np.float32)
    ext["humidity"]         = df_w["humidity"].values[:n_t].astype(np.float32)
    ext["precipitation"]    = df_w["precipitation"].values[:n_t].astype(np.float32)
    ext["temperature_norm"] = _norm_w(ext["temperature"])
    ext["humidity_norm"]    = _norm_w(ext["humidity"])
    WEATHER_AVAILABLE = True
    print(f"[Ext] Weather OK")
except FileNotFoundError:
    print("[Ext] weather_shenzhen.csv no encontrado — sin datos meteo")

# Addresses
try:
    df_a = pd.read_csv(f"{DATASETS_PATH}/addresses_shenzhen.csv")
    ext["addresses"] = df_a.set_index("zone_id")["address_short"].to_dict()
    ADDRESS_AVAILABLE = True
    print(f"[Ext] Addresses OK ({len(df_a)} zonas)")
except FileNotFoundError:
    print("[Ext] addresses_shenzhen.csv no encontrado")

# Spatial
try:
    df_sp = pd.read_csv(f"{DATASETS_PATH}/spatial_features_shenzhen.csv")
    ext["road_length"] = df_sp.set_index("zone_id")["road_length_km"].to_dict()
    ext["poi_count"]   = df_sp.set_index("zone_id")["poi_count"].to_dict()
    SPATIAL_AVAILABLE = True
    print(f"[Ext] Spatial OK")
except FileNotFoundError:
    print("[Ext] spatial_features_shenzhen.csv no encontrado")

# Calendar
try:
    df_cal = pd.read_csv(f"{DATASETS_PATH}/calendar_features.csv")
    n_t = meta["n_pasos"]
    ext["cal_is_weekend"] = df_cal["is_weekend"].values[:n_t]
    ext["cal_is_holiday"] = df_cal["is_holiday"].values[:n_t]
    ext["cal_hol_name"]   = df_cal["holiday_name"].fillna("").values[:n_t]
    ext["cal_is_peak"]    = df_cal["is_peak_hour"].values[:n_t]
    ext["cal_period"]     = df_cal["time_period"].values[:n_t]
    ext["cal_day_name"]   = df_cal["day_name"].values[:n_t]
    CALENDAR_AVAILABLE = True
    print(f"[Ext] Calendar OK")
except FileNotFoundError:
    print("[Ext] calendar_features.csv no encontrado")

print(f"\nResumen: weather={WEATHER_AVAILABLE} | address={ADDRESS_AVAILABLE} "
      f"| spatial={SPATIAL_AVAILABLE} | calendar={CALENDAR_AVAILABLE}")

[Ext] Weather OK
[Ext] Addresses OK (247 zonas)
[Ext] spatial_features_shenzhen.csv no encontrado
[Ext] Calendar OK

Resumen: weather=True | address=True | spatial=False | calendar=True


In [21]:
# ── Funciones de inferencia y construcción de prompts ──────────────────────

def predecir_numeric(model, tokenizer, texto_inputs: list, max_new: int = 10) -> np.ndarray:
    preds = []
    for txt in texto_inputs:
        enc = tokenizer(txt, return_tensors="pt", truncation=True, max_length=512).to(DEVICE)
        with torch.no_grad():
            ids = model.generate(**enc, max_new_tokens=max_new)
        out = tokenizer.decode(ids[0], skip_special_tokens=True).strip()
        try:
            preds.append(float(out))
        except ValueError:
            nums = re.findall(r"[-+]?\d*\.?\d+", out)
            preds.append(float(nums[0]) if nums else 0.0)
    return np.array(preds)


def descripcion_area(info_row: pd.Series, zona_id: int) -> str:
    def safe(col, fmt="{}", default="N/A"):
        if col in info_row.index:
            v = info_row[col]
            return fmt.format(v) if pd.notna(v) else default
        return default
    count = safe("count", "{:.0f}")
    fast  = safe("fast_count", "{:.0f}")
    slow  = safe("slow_count", "{:.0f}")
    cbd   = "Central Business District" if str(safe("cbd")) == "1.0" else "Non-CBD"
    pric  = "dynamic pricing" if str(safe("dynamic_pricing")) == "1.0" else "flat-rate pricing"
    lon   = safe("lon", "{:.4f}")
    lat   = safe("la",  "{:.4f}")
    return (f"Area ID={zona_id}, Coordinates=[{lat}N, {lon}E], "
            f"Area Type={cbd}, Total Piles={count} ({fast} fast / {slow} slow), "
            f"Pricing={pric}")


def descripcion_area_v2(info_row: pd.Series, zona_id: int, t_step: int = None) -> str:
    def safe(col, fmt="{}", default="N/A"):
        if col in info_row.index:
            v = info_row[col]
            return fmt.format(v) if pd.notna(v) else default
        return default
    count = safe("count", "{:.0f}")
    fast  = safe("fast_count", "{:.0f}")
    slow  = safe("slow_count", "{:.0f}")
    cbd   = "Central Business District" if str(safe("cbd")) == "1.0" else "Non-CBD"
    pric  = "dynamic pricing" if str(safe("dynamic_pricing")) == "1.0" else "flat-rate pricing"
    lon   = safe("lon", "{:.4f}")
    lat   = safe("la",  "{:.4f}")
    parts = [f"Area ID={zona_id}", f"Coordinates=[{lat}N, {lon}E]"]
    if ADDRESS_AVAILABLE:
        addr = ext["addresses"].get(zona_id, ext["addresses"].get(str(zona_id), "N/A"))
        parts.append(f"Address={addr}")
    parts += [f"Area Type={cbd}",
              f"Total Piles={count} ({fast} fast / {slow} slow)",
              f"Pricing={pric}"]
    if SPATIAL_AVAILABLE:
        road = ext["road_length"].get(zona_id, ext["road_length"].get(str(zona_id), None))
        poi  = ext["poi_count"].get(zona_id, ext["poi_count"].get(str(zona_id), None))
        if road is not None: parts.append(f"Road Length={road:.2f} km")
        if poi  is not None: parts.append(f"POI Count={int(poi)}")
    if WEATHER_AVAILABLE and t_step is not None:
        t    = min(t_step, len(ext["temperature"]) - 1)
        temp = ext["temperature"][t]
        hum  = ext["humidity"][t]
        rain = "rainy" if ext["precipitation"][t] > 0.1 else "dry"
        parts.append(f"Weather=[{temp:.1f}C, humidity {hum:.0f}%, {rain}]")
    if CALENDAR_AVAILABLE and t_step is not None:
        t      = min(t_step, len(ext["cal_day_name"]) - 1)
        dname  = ext["cal_day_name"][t]
        period = ext["cal_period"][t]
        dtype  = ("Holiday" if ext["cal_is_holiday"][t]
                  else "Weekend" if ext["cal_is_weekend"][t] else "Weekday")
        peak   = "peak hour" if ext["cal_is_peak"][t] else "off-peak"
        parts.append(f"Time=[{dname} {period}, {dtype}, {peak}]")
    return ", ".join(parts)


def _fmt(arr):
    return "[" + ", ".join(f"{v:.2f}" for v in arr) + "]"


def construir_prompt(zona_id, info_row, local_occ, vecino_occ,
                     precio_w, duracion_w, t_step=None):
    use_v2 = any([WEATHER_AVAILABLE, ADDRESS_AVAILABLE,
                  SPATIAL_AVAILABLE, CALENDAR_AVAILABLE])
    if use_v2:
        area_desc = descripcion_area_v2(info_row, zona_id, t_step)
        weather_lines = ""
        if WEATHER_AVAILABLE and t_step is not None:
            lw     = max(0, t_step - len(local_occ))
            temp_w = ext["temperature_norm"][lw: t_step]
            hum_w  = ext["humidity_norm"][lw: t_step]
            if len(temp_w) == len(local_occ):
                weather_lines = (f"Temperature (norm)={_fmt(temp_w)}; "
                                 f"Humidity (norm)={_fmt(hum_w)}. ")
    else:
        area_desc     = descripcion_area(info_row, zona_id)
        weather_lines = ""

    return (
        "You are an expert in electric vehicle charging management, "
        "who is good at <charging demand prediction>. "
        f"We are now in {area_desc}. "
        "Given the following time series of historical charging data, "
        f"Local Charging Occupancy={_fmt(local_occ)}; "
        f"Average Neighboring Charging Occupancy={_fmt(vecino_occ)}; "
        f"Charging price={_fmt(precio_w)}; "
        f"Charging Duration={_fmt(duracion_w)}. "
        + weather_lines +
        "Now, pay attention! Your task is to <predict the charging demand "
        "in the area for the next hour> by analyzing the given information "
        "and leveraging your common sense. "
        "In your answer, you should provide the value of your prediction only."
    )


def chatev_predecir_zona(zona_id: int, t_step: int = None) -> tuple:
    if not (0 <= zona_id < n_zonas):
        return f"Zona ID debe estar entre 0 y {n_zonas-1}.", ""

    ts_test, te_test = time_div["test"]
    if t_step is None:
        t_step = ts_test + LOOKBACK_W + (te_test - ts_test - LOOKBACK_W - HORIZON) // 2
    t_step = int(np.clip(t_step, ts_test + LOOKBACK_W, te_test - HORIZON - 1))

    local_w = occ_norm[zona_id,   t_step - LOOKBACK_W: t_step]
    vec_w   = vecino_occ[zona_id, t_step - LOOKBACK_W: t_step]
    prec_w  = price_norm[zona_id, t_step - LOOKBACK_W: t_step]
    dur_w   = dur_norm[zona_id,   t_step - LOOKBACK_W: t_step]
    true_val = float(np.mean(occ_norm[zona_id, t_step: t_step + HORIZON]))

    row    = info.iloc[zona_id] if zona_id < len(info) else info.iloc[0]
    prompt = construir_prompt(zona_id, row, local_w, vec_w, prec_w, dur_w, t_step)
    pred   = float(predecir_numeric(model, tokenizer, [prompt])[0])

    try:
        ts_label = str(timestamps[t_step])[:16]
    except Exception:
        ts_label = f"t={t_step}"

    zona_tipo = "VISTA en entrenamiento" if zona_id in seen_zonas else "NO VISTA (zero-shot)"
    trend = "↑ subiendo" if local_w[-1] > local_w[0] else "↓ bajando"

    respuesta = (
        f"### Zona {zona_id} — {ts_label}\n\n"
        f"🏷️ {zona_tipo}\n\n"
        f"**Predicción ChatEV (próximos 30 min): `{pred:.2f}`**\n\n"
        f"| | Valor |\n|---|---|\n"
        f"| Predicción | `{pred:.4f}` |\n"
        f"| Valor real (test) | `{true_val:.4f}` |\n"
        f"| Error absoluto | `{abs(pred - true_val):.4f}` |\n"
        f"| Ocupación reciente (media 3 pasos) | `{float(local_w[-3:].mean()):.4f}` |\n"
        f"| Tendencia | {trend} |\n\n"
        f"*Ventana: {LOOKBACK_W} pasos (1h) · Horizonte: {HORIZON} pasos (30min)*"
    )
    return respuesta, prompt


def _parse_zona(msg: str) -> tuple:
    t_m = re.search(r'(?:step|t)[=\s]+(\d+)', msg, re.IGNORECASE)
    t_s = int(t_m.group(1)) if t_m else None
    z_m = re.search(r'(?:zone|zona|z)[=\s]+(\d+)', msg, re.IGNORECASE)
    if z_m:
        return int(z_m.group(1)), t_s
    nums = re.findall(r'\b(\d{1,3})\b', msg)
    return (int(nums[0]) if nums else -1), t_s


def chatev_chat(message: str, history: list) -> str:
    msg = message.strip()
    if any(k in msg.lower() for k in ["help", "ayuda", "?", "cómo", "como"]):
        return (
            "**ChatEV — Instrucciones**\n\n"
            "Consulta la predicción de demanda EV para cualquier zona:\n"
            "- `zona 42` o `zone 42`\n"
            "- `zona 42 step 7500` (timestep concreto)\n"
            "- Solo número: `42`\n\n"
            f"Dataset: {n_zonas} zonas · Shenzhen 2022\n"
            f"Modelo: T5-small + Reptile meta-learning\n"
            f"Datos externos: weather={WEATHER_AVAILABLE} · "
            f"address={ADDRESS_AVAILABLE} · spatial={SPATIAL_AVAILABLE} · "
            f"calendar={CALENDAR_AVAILABLE}"
        )
    zona_id, t_step = _parse_zona(msg)
    if zona_id < 0 or zona_id >= n_zonas:
        return (f"❌ Zona no válida. Especifica 0–{n_zonas-1}.\n"
                "Ejemplo: `zona 42` o `42`. Escribe `help` para más opciones.")
    respuesta, prompt = chatev_predecir_zona(zona_id, t_step)
    preview = prompt[:400] + "..." if len(prompt) > 400 else prompt
    respuesta += (f"\n\n---\n<details><summary>🔍 Ver prompt interno</summary>"
                  f"\n\n```\n{preview}\n```\n</details>")
    return respuesta


print("Funciones cargadas. Listo para lanzar Gradio.")

Funciones cargadas. Listo para lanzar Gradio.


In [22]:
demo = gr.ChatInterface(
    fn=chatev_chat,
    title="💬 ChatEV — EV Charging Demand Predictor",
    description=(
        "Predice demanda de recarga EV en lenguaje natural.\n"
        "Escribe **`zona <ID>`** (0–246) o el número directamente. "
        "Escribe `help` para instrucciones.\n"
        "*T5-small · Reptile · ST-EVCDP Shenzhen 2022*"
    ),
    examples=[["zona 42"],["zone 100"],["zona 42 step 7500"],["help"]],
    theme=gr.themes.Soft(),
    chatbot=gr.Chatbot(height=450, render_markdown=True),
)
demo.launch(share=True, debug=False)

/tmp/ipykernel_27611/2231882264.py:12: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot=gr.Chatbot(height=450, render_markdown=True),
/tmp/ipykernel_27611/2231882264.py:12: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot=gr.Chatbot(height=450, render_markdown=True),
/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:330: UserWarning: The gr.ChatInterface was not provided with a type, so the type of the gr.Chatbot, 'tuples', will be used.
  warnings.warn(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://2ead0c196ed9d053a3.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
